# Data Cleaning and Preparation — Hands-on

This notebook follows the **Building Datasets** lecture. We load raw text from a source, run it through a cleaning pipeline, and compare **before/after** stats (document count, token count, duplicate rate). The same pipeline is then applied to a second source to show **reusability**.

**Data cleaning essentials covered:**

1. **Text-specific handling**: Encoding (e.g. UTF-8), normalization (Unicode, lowercasing, accents).
2. **Noise reduction**: Stripping HTML/XML tags, boilerplate, and broken markup.
3. **Deduplication**: Exact deduplication plus an optional near-duplicate step.
4. **Missing or malformed data**: Dropping empty and too-short documents.
5. **Bias and safety**: Optional PII redaction and notes on toxic/sensitive filtering.
6. **Reproducibility**: A single, scripted pipeline with clear steps so runs are repeatable.

## Learning objectives

- Run one **end-to-end** cleaning pipeline: raw → cleaned dataset.
- Interpret **before/after** stats (doc count, token count, duplicate %).
- Reuse the **same pipeline** on another source.
- See how each of the six data-cleaning essentials is applied in code.

## Setup and dependencies

**Prerequisites:** Python 3.8+, pip.

Run the cell below only if `pandas` and `datasets` are not already installed.

In [ ]:
import re
import os
import unicodedata
import pandas as pd
from datasets import load_dataset

## Helper: computing stats

We track **document count**, **token count** (approximate, by word split), and **duplicate rate** (share of documents that are exact duplicates of another). These metrics let us compare before and after cleaning.

In [ ]:
def get_stats(texts):
    """Return doc count, total tokens (word split), and duplicate rate."""
    if not texts:
        return {"doc_count": 0, "token_count": 0, "duplicate_pct": 0.0}
    doc_count = len(texts)
    token_count = sum(len(t.split()) for t in texts)
    unique = len(set(texts))
    duplicate_pct = (1 - unique / doc_count) * 100 if doc_count else 0.0
    return {"doc_count": doc_count, "token_count": token_count, "duplicate_pct": duplicate_pct}


def print_comparison_table(before_stats, after_stats, label_before="Before", label_after="After"):
    """Print a small before/after comparison table."""
    df = pd.DataFrame([before_stats, after_stats], index=[label_before, label_after])
    print(df.to_string())
    return df

## Reusable cleaning pipeline

Pipeline steps (aligned with the six data-cleaning essentials):

1. **Text-specific handling**: Decode as UTF-8, normalize Unicode (NFKC), lowercase, normalize accents.
2. **Noise reduction**: Remove HTML/XML tags and collapse extra whitespace.
3. **Deduplication**: Drop empty → exact deduplicate (optionally near-duplicate via simple n-gram hashing).
4. **Missing/malformed**: Drop empty strings and documents shorter than `min_length` words.
5. **Bias and safety**: Optional PII redaction (e.g. email, phone-like patterns); toxic/sensitive filtering can be added similarly.
6. **Reproducibility**: One function, fixed order of steps, so the pipeline is scriptable and repeatable.

In [ ]:
# PII redaction patterns (simple regex; extend as needed for your use case)
PII_EMAIL = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
PII_PHONE = re.compile(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b")


def _normalize_text(s):
    """(1) Text-specific: encoding + Unicode normalization + lowercasing."""
    if not isinstance(s, str):
        s = s.encode("utf-8", errors="replace").decode("utf-8")
    s = unicodedata.normalize("NFKC", s).strip().lower()
    return s


def _strip_html(text):
    """(2) Noise reduction: remove HTML/XML tags and collapse whitespace."""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _redact_pii(text, redact_email=True, redact_phone=True):
    """(5) Bias & safety: optional PII redaction."""
    if redact_email:
        text = PII_EMAIL.sub("[EMAIL]", text)
    if redact_phone:
        text = PII_PHONE.sub("[PHONE]", text)
    return text


def _near_dup_hash(text, n=5):
    """Simple n-gram hash for optional near-duplicate detection (no extra deps)."""
    t = " " + _normalize_text(text) + " "
    shingles = [t[i : i + n] for i in range(len(t) - n + 1)]
    return hash(frozenset(shingles))


def clean_text_pipeline(
    texts,
    min_length=10,
    strip_html=True,
    redact_pii=False,
    remove_near_duplicates=False,
):
    """
    One reproducible pipeline: normalize → strip HTML → drop empty → exact dedup
    → optional PII redaction → optional near-dedup → filter by min_length.
    Returns cleaned list of strings.
    """
    out = []
    for t in texts:
        if t is None or (isinstance(t, str) and not t.strip()):
            continue
        s = _normalize_text(t)
        if strip_html:
            s = _strip_html(s)
        if not s:
            continue
        if redact_pii:
            s = _redact_pii(s)
        out.append(s)

    # Exact deduplication (3), preserving order
    out = list(dict.fromkeys(out))

    # Optional near-duplicate removal (3): keep first of each n-gram hash
    if remove_near_duplicates:
        seen_hashes = set()
        deduped = []
        for t in out:
            h = _near_dup_hash(t)
            if h not in seen_hashes:
                seen_hashes.add(h)
                deduped.append(t)
        out = deduped

    # (4) Missing/malformed: filter by minimum word count
    out = [t for t in out if len(t.split()) >= min_length]
    return out

## Source 1: Load raw data (IMDB)

We use a small subset of the [IMDB](https://huggingface.co/datasets/imdb) dataset from Hugging Face. No API key required.

In [ ]:
ds = load_dataset("imdb", split="train[:500]", trust_remote_code=True)
raw_texts_1 = [item["text"] for item in ds]
print(f"Loaded {len(raw_texts_1)} documents.")
stats_before_1 = get_stats(raw_texts_1)
print("Before cleaning:")
print(f"  Doc count: {stats_before_1['doc_count']}, Token count: {stats_before_1['token_count']}, Duplicate %: {stats_before_1['duplicate_pct']:.2f}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imdb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
d:\Prajwal\Jnana\LLM Course\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Prajwal\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In

Loaded 500 documents.
Before cleaning:
  Doc count: 500, Token count: 110706, Duplicate %: 0.20


## Run pipeline on Source 1

Apply the cleaning pipeline (normalize, strip HTML, drop empty, deduplicate, filter by min length). Optionally enable `redact_pii=True` or `remove_near_duplicates=True` to demonstrate those steps.

In [ ]:
cleaned_1 = clean_text_pipeline(
    raw_texts_1,
    min_length=10,
    strip_html=True,
    redact_pii=False,
    remove_near_duplicates=False,
)
stats_after_1 = get_stats(cleaned_1)
print("After cleaning:")
print(f"  Doc count: {stats_after_1['doc_count']}, Token count: {stats_after_1['token_count']}, Duplicate %: {stats_after_1['duplicate_pct']:.2f}")

After cleaning:
  Doc count: 499, Token count: 109194, Duplicate %: 0.00


## Before/after comparison (Source 1)

Summary: we removed empty and short docs, duplicates, and noise; final doc count and token count reflect the cleaned dataset.

In [ ]:
print_comparison_table(stats_before_1, stats_after_1, label_before="Before", label_after="After")

        doc_count  token_count  duplicate_pct
Before        500       110706            0.2
After         499       109194            0.0


,doc_count,token_count,duplicate_pct
Before,500,110706,0.2
After,499,109194,0.0


## Save cleaned dataset

Write the cleaned list to disk (one document per line) for reuse. The `data/` folder is created if it does not exist.

In [ ]:
os.makedirs("data", exist_ok=True)
out_path_1 = "data/cleaned_source1.txt"
with open(out_path_1, "w", encoding="utf-8") as f:
    for line in cleaned_1:
        f.write(line.strip() + "\n")
print(f"Saved {len(cleaned_1)} documents to {out_path_1}")

Saved 499 documents to data/cleaned_source1.txt


## Optional: Source 2 — pipeline reusability

The same `clean_text_pipeline` is applied to a different dataset ([AG News](https://huggingface.co/datasets/ag_news)) to show that one scripted pipeline works across sources.

In [ ]:
ds2 = load_dataset("ag_news", split="train[:500]", trust_remote_code=True)
raw_texts_2 = [item["text"] for item in ds2]
stats_before_2 = get_stats(raw_texts_2)
cleaned_2 = clean_text_pipeline(raw_texts_2, min_length=10, strip_html=True)
stats_after_2 = get_stats(cleaned_2)
print("Source 2 (AG News) — Before:", stats_before_2)
print("Source 2 (AG News) — After:", stats_after_2)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
d:\Prajwal\Jnana\LLM Course\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Prajwal\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrat

Source 2 (AG News) — Before: {'doc_count': 500, 'token_count': 20474, 'duplicate_pct': 0.0}
Source 2 (AG News) — After: {'doc_count': 500, 'token_count': 20474, 'duplicate_pct': 0.0}


In [ ]:
comparison = pd.DataFrame({
    "Source": ["IMDB (1)", "AG News (2)"],
    "Dup % before": [stats_before_1["duplicate_pct"], stats_before_2["duplicate_pct"]],
    "Dup % after": [stats_after_1["duplicate_pct"], stats_after_2["duplicate_pct"]],
})
print(comparison.to_string(index=False))

# Optionally save Source 2 the same way
with open("data/cleaned_source2.txt", "w", encoding="utf-8") as f:
    for line in cleaned_2:
        f.write(line.strip() + "\n")
print(f"\nSaved {len(cleaned_2)} documents to data/cleaned_source2.txt")

     Source  Dup % before  Dup % after
   IMDB (1)           0.2          0.0
AG News (2)           0.0          0.0

Saved 500 documents to data/cleaned_source2.txt


## Wrap-up

- **One pipeline**: normalize → strip HTML → drop empty → exact deduplicate → filter by `min_length` (plus optional PII redaction and near-dedup).
- **Same code** for multiple sources: reuse `clean_text_pipeline` on any list of text strings.
- **Measure before/after**: doc count, token count, and duplicate % tell you how much noise was removed and help with reproducibility and quality control.
- The six data-cleaning essentials (text handling, noise reduction, deduplication, missing/malformed data, bias & safety, reproducibility) are all reflected in this single, scripted pipeline.